# DCASE 2026 Task 1

**Tema:** Heterogena klasifikacija zvuka pomoću Broad Sound Taxonomy (BST)

**Korišćeno rešenje:** zvanični baseline za DCASE 2026 Task 1

U radu je prikazana primena zvaničnog baseline sistema na skupu BSD10k-v1.2, od pripreme podataka do evaluacije modela. Posebna pažnja posvećena je uticaju pouzdanosti anotacija na uspešnost klasifikacije zvuka.

## 1. Cilj projekta i istraživačko pitanje

DCASE 2026 Task 1 obuhvata klasifikaciju proizvoljnih zvučnih snimaka u široke kategorije zvuka. Klasifikacija se zasniva na hijerarhijskoj taksonomiji BST, koja sadrži pet kategorija najvišeg i 23 kategorije drugog nivoa.

Glavno istraživačko pitanje projekta glasi:

> Da li manji, ali pouzdanije anotiran skup daje bolji rezultat od kompletnog skupa?

Zvanični baseline najpre je reprodukovan kako bi njegovi rezultati poslužili kao referentna tačka za poređenje različitih varijanti skupa podataka.

## 2. Podaci i reprezentacija

U eksperimentima je korišćen **BSD10k-v1.2**, kurirani Freesound skup sa ekspertskim anotacijama, koji nakon pripreme sadrži 10.956 primera. Svaki primer predstavljen je pomoću dve unapred izračunate reprezentacije dimenzije 512:

- audio embedding dobijen modelom LAION-CLAP;
- tekstualni embedding opisa zvuka, takođe dobijen modelom LAION-CLAP.

Skripta `build_dataset.py` objedinjuje metapodatke, putanje do embeddinga, oznake kategorija i njihove numeričke indekse u datoteci `processed_dataset.csv` i odgovarajućim JSON rečnicima. Audio i tekstualne reprezentacije dostupne su za svih 10.956 primera.

In [1]:
from pathlib import Path
import csv

PROJECT_DIR = Path.cwd()
BASELINE_DIR = PROJECT_DIR / "dcase2026_task1_baseline"
if not BASELINE_DIR.exists():
    BASELINE_DIR = PROJECT_DIR

dataset_csv = BASELINE_DIR / "data" / "processed_dataset.csv"
with dataset_csv.open(encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print(f"Broj pripremljenih primera: {len(rows):,}".replace(",", "."))
print(f"Broj kategorija drugog nivoa: {len({row['class'] for row in rows})}")
print(f"Broj kategorija najvišeg nivoa: {len({row['top_class'] for row in rows})}")

Broj pripremljenih primera: 10.956
Broj kategorija drugog nivoa: 23
Broj kategorija najvišeg nivoa: 5


## 3. Baseline model

Baseline predstavlja varijantu HATR modela i primenjen je u dva režima:

1. **Audio-only** — klasifikacija samo na osnovu audio embeddinga.
2. **Multimodalni (`both`)** — koristi audio i tekstualni embedding.

Model sadrži odvojene enkodere modaliteta, mehanizam pažnje za njihovu fuziju i klasifikator sa rezidualnim blokovima. Tokom treniranja koriste se augmentacije embeddinga: Gausov šum i nasumično maskiranje. Model direktno predviđa jednu od 23 kategorije drugog nivoa, dok se pripadnost kategoriji najvišeg nivoa izvodi iz BST hijerarhije.

Evaluacija obuhvata standardne i hijerarhijske metrike. Kao glavna metrika izdvaja se **hierarchical F1**, pošto istovremeno uzima u obzir predviđenu kategoriju i njen položaj u hijerarhiji.

## 4. Priprema eksperimentalnog okruženja

Za izvođenje eksperimenata pripremljeno je lokalno Python okruženje sa potrebnim bibliotekama, metapodacima i CLAP reprezentacijama skupa BSD10k-v1.2. Eksperimentalni tok obuhvata učitavanje podataka, treniranje, izbor najboljeg checkpointa, testiranje i čuvanje predikcija.

Skripta `train_test.py` prilagođena je tako da se režim rada, izlazni direktorijum, broj epoha, najveći broj foldova i broj DataLoader procesa zadaju promenljivama okruženja. Ovakva postavka omogućava izvođenje kraćih provera bez izmene osnovnih hiperparametara u kodu.

Primer pokretanja kontrolisanog eksperimenta:

```bash
DCASE_MODES=audio,both \
DCASE_MODEL_OUTPUT=./model_output_medium \
DCASE_NUM_EPOCHS=10 \
DCASE_K_FOLDS=5 \
DCASE_MAX_FOLDS=1 \
DCASE_NUM_WORKERS=0 \
.venv/bin/python train_test.py
```

## 5. Provera eksperimentalnog toka

Pre punog treninga izvedene su dve kraće provere eksperimentalne postavke.

### 5.1. Početna provera

Oba režima trenirana su tokom dve epohe na jednom foldu. Dobijeni rezultati nisu korišćeni kao konačni, već za proveru ispravnosti svih faza eksperimenta.

| Režim | Accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---:|---:|---:|
| Audio | 64,87% | 60,31% | 65,44% |
| Audio + tekst | 72,67% | 68,58% | 69,07% |

### 5.2. Proširena provera

U narednoj proveri oba režima trenirana su tokom deset epoha na jednom foldu. Validaciona tačnost rasla je stabilno, a najbolji model pravilno je sačuvan.

| Režim | Accuracy | Macro accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---:|---:|---:|---:|
| Audio | 75,96% | 64,83% | 71,49% | 72,24% |
| Audio + tekst | 77,69% | 66,75% | 72,98% | 72,67% |

Najbolja validaciona tačnost za oba modela ostvarena je u osmoj epohi: 76,16% za audio i 78,61% za multimodalni model.

## 6. Puni 5-fold eksperiment

Puni eksperiment za oba režima izveden je stratifikovanim unakrsnim testiranjem sa pet foldova. U svakom foldu približno 20% podataka pripada test skupu, dok je deo preostalih podataka izdvojen za validaciju.

Maksimalan broj epoha bio je 100, uz prevremeno zaustavljanje. Treniranje audio modela završavalo se između 30. i 49. epohe, a multimodalnog između 32. i 42. epohe. Za svaki fold sačuvani su:

- najbolji model (`best_model.pth`);
- istorija treniranja (`history.json`);
- podela primera (`splits.csv`);
- test predikcije (`predictions.csv`);
- evaluacione metrike (`results.txt`).

In [2]:
import re
from collections import defaultdict
from statistics import mean, pstdev

results_root = BASELINE_DIR / "model_output_full"
pattern = re.compile(r"([a-zA-Z0-9_]+):\s*([\d.]+)%")
metrics = defaultdict(lambda: defaultdict(list))

for result_file in sorted(results_root.glob("*/fold_*/evaluation/results.txt")):
    mode = result_file.parts[-4]
    for name, value in pattern.findall(result_file.read_text(encoding="utf-8")):
        metrics[mode][name].append(float(value))

for mode in ("audio", "both"):
    print(f"{mode}:")
    for metric in ("accuracy", "macro_accuracy", "hierarchical_accuracy", "hierarchical_f1"):
        values = metrics[mode][metric]
        print(f"  {metric}: {mean(values):.2f}% ± {pstdev(values):.2f}%")

audio:
  accuracy: 77.09% ± 0.64%
  macro_accuracy: 69.88% ± 1.36%
  hierarchical_accuracy: 76.49% ± 0.96%
  hierarchical_f1: 75.20% ± 0.90%
both:
  accuracy: 79.98% ± 0.66%
  macro_accuracy: 74.32% ± 0.77%
  hierarchical_accuracy: 79.66% ± 0.75%
  hierarchical_f1: 78.65% ± 0.61%


### 6.1. Zbirni rezultati

| Režim | Accuracy | Macro accuracy | Top-level accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---:|---:|---:|---:|---:|
| Audio | 77,09% ± 0,64% | 69,88% ± 1,36% | 87,71% ± 0,28% | 76,49% ± 0,96% | 75,20% ± 0,90% |
| Audio + tekst | **79,98% ± 0,66%** | **74,32% ± 0,77%** | **89,18% ± 0,68%** | **79,66% ± 0,75%** | **78,65% ± 0,61%** |

Uvođenjem tekstualne reprezentacije tačnost je povećana za 2,89, makro tačnost za 4,44, hijerarhijska tačnost za 3,17, a hierarchical F1 za 3,45 procentnih poena. Pored boljeg prosečnog rezultata, multimodalni model ima i manju standardnu devijaciju za hierarchical F1 (0,61 naspram 0,90), pa su njegovi rezultati nešto ujednačeniji između foldova.

In [3]:
for mode in ("audio", "both"):
    values = metrics[mode]["hierarchical_f1"]
    formatted = ", ".join(f"{value:.2f}%" for value in values)
    print(f"{mode}: {formatted}")

audio: 75.51%, 73.67%, 76.14%, 75.92%, 74.78%
both: 78.13%, 78.95%, 78.30%, 78.16%, 79.71%


### 6.2. Poređenje sa objavljenim baseline rezultatima

| Režim | Naš hierarchical accuracy | Objavljeno | Naš hierarchical F1 | Objavljeno |
|---|---:|---:|---:|---:|
| Audio | 76,49% | 77,36% | 75,20% | 76,11% |
| Audio + tekst | 79,66% | 79,71% | 78,65% | 78,76% |

Rezultat multimodalnog modela gotovo se podudara sa objavljenim vrednostima: razlika iznosi 0,05 poena za hierarchical accuracy i 0,11 poena za hierarchical F1. Kod audio modela razlika je približno 0,9 poena, što je i dalje u očekivanom opsegu. Time je potvrđena uspešna reprodukcija baseline sistema, dok se manja odstupanja mogu pripisati razlikama u hardveru, verzijama biblioteka i nedeterminističkim numeričkim operacijama.

## 7. Eksperiment sa pouzdanijim anotacijama

Pouzdaniji podskup formiran je od primera za koje važi `confidence >= 4`. Ovaj prag predstavlja prihvatljiv odnos između kvaliteta anotacija i veličine skupa: zadržano je 6.820 od ukupno 10.956 primera (62,25%), pri čemu su očuvane sve 23 kategorije, a najmanja među njima sadrži 72 primera. Stroži prag 5 sveo bi skup na samo 777 primera i izostavio jednu kategoriju.

Filtriranje je izvedeno skriptom `prepare_confident_subset.py`. Udeo zadržanih primera nije jednak za sve klase i kreće se od 30,30% do 86,74%, zbog čega makro metrike treba tumačiti uz izvesnu rezervu. Na ovom skupu ponovljeno je unakrsno testiranje sa pet foldova za oba režima, uz iste hiperparametre i postupak prevremenog zaustavljanja.

| Skup | Režim | Accuracy | Macro accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---|---:|---:|---:|---:|
| Kompletan (10.956) | Audio | 77,09% ± 0,64% | 69,88% ± 1,36% | 76,49% ± 0,96% | 75,20% ± 0,90% |
| Pouzdaniji (6.820) | Audio | **85,62% ± 1,04%** | **75,79% ± 2,01%** | **81,21% ± 1,53%** | **80,06% ± 3,02%** |
| Kompletan (10.956) | Audio + tekst | 79,98% ± 0,66% | 74,32% ± 0,77% | 79,66% ± 0,75% | 78,65% ± 0,61% |
| Pouzdaniji (6.820) | Audio + tekst | **87,86% ± 0,38%** | **79,64% ± 0,73%** | **83,86% ± 0,63%** | **83,01% ± 0,47%** |

Na filtriranom skupu hierarchical F1 veći je za 4,86 poena u audio režimu i za 4,36 poena u multimodalnom režimu. Rezultati multimodalnog modela pritom su veoma ujednačeni između foldova. Veća standardna devijacija audio modela posledica je jednog slabijeg folda sa rezultatom od 74,38%, što ukazuje na osetljivost ređih klasa i neujednačene raspodele nakon filtriranja.

Ovakvo poređenje ipak ima važno ograničenje, jer modeli trenirani na kompletnom i filtriranom skupu nisu evaluirani na istim test primerima. Test deo filtriranog skupa sadrži samo pouzdanije anotacije, pa može predstavljati lakši problem. Zbog toga je sproveden i kontrolisani eksperiment sa zajedničkim test skupom.

## 8. Kontrolisani eksperiment sa zajedničkim test skupom

Iz kompletnog skupa izdvojen je stratifikovani test skup sa 2.192 primera, odnosno 20% podataka. Isti test primeri korišćeni su za svih šest modela i nisu učestvovali ni u treniranju ni u izboru checkpointa. Poređene su tri varijante trening skupa: svi preostali primeri (7.011 za treniranje i 1.753 za validaciju), primeri sa `confidence >= 4` (4.320 i 1.081) i nasumični podskup iste veličine i klasne raspodele kao pouzdani podskup.

| Trening skup | Režim | Accuracy | Macro accuracy | Hierarchical accuracy | Hierarchical F1 |
|---|---|---:|---:|---:|---:|
| Kompletan | Audio | **78,19%** | **72,18%** | **78,10%** | **76,83%** |
| `confidence >= 4` | Audio | 78,01% | 71,17% | 77,68% | 75,91% |
| Nasumični podskup | Audio | 76,19% | 66,57% | 73,48% | 71,95% |
| Kompletan | Audio + tekst | **80,98%** | **75,82%** | **81,04%** | **79,78%** |
| `confidence >= 4` | Audio + tekst | 78,88% | 72,69% | 78,91% | 77,11% |
| Nasumični podskup | Audio + tekst | 80,16% | 73,44% | 79,06% | 77,63% |

Na zajedničkom test skupu manji pouzdani skup nije nadmašio kompletan skup. Razlika u metrici hierarchical F1 iznosi 0,92 poena u audio režimu i 2,67 poena u multimodalnom režimu. Sa druge strane, audio model treniran na pouzdanom podskupu nadmašio je model treniran na nasumičnom podskupu iste veličine za 3,96 poena. Pažljiv izbor anotacija, dakle, može da ublaži gubitak nastao smanjenjem broja trening primera. U multimodalnom režimu nasumični podskup ostvario je za 0,52 poena bolji rezultat od pouzdanog.

Rezultati zato ne daju potvrdan odgovor na početno istraživačko pitanje. Pouzdanije anotacije mogu biti vrednije od iste količine nasumično izabranih podataka, naročito kod audio modela, ali je veća količina podataka iz kompletnog skupa ipak omogućila najbolju generalizaciju.

## 9. Zaključak

U ovom radu uspešno je reprodukovan zvanični DCASE 2026 Task 1 baseline na skupu BSD10k-v1.2. Rezultati potvrđuju značaj tekstualne reprezentacije opisa zvuka: multimodalni model ostvario je hierarchical F1 od 78,65% ± 0,61%, što je za 3,45 procentnih poena više od audio modela i veoma blizu zvanično objavljenom rezultatu.

Prvobitno poređenje pokazalo je bolje rezultate na podskupu sa pouzdanijim anotacijama, ali takav rezultat nije dovoljan za zaključak da je manji skup bolji, jer je i njegov test deo filtriran. Kada su svi modeli evaluirani na istom test skupu, najbolje rezultate ostvarili su modeli trenirani na kompletnom skupu: hierarchical F1 iznosio je 76,83% za audio i 79,78% za multimodalni režim.

Ipak, pouzdanost anotacija nije bez značaja. U audio režimu pažljivo izabran podskup bio je za 3,96 procentnih poena bolji od nasumičnog podskupa iste veličine, iako je za 0,92 poena zaostajao za kompletnim skupom. Na osnovu toga može se zaključiti da kvalitet anotacija ublažava posledice manjeg broja primera, ali u ovom slučaju ne nadoknađuje u potpunosti prednost koju donosi veća količina podataka.